# MangaForge AI — Kaggle Dataset Pipeline

**Commercial-use safety:** Openverse metadata is only a lead. CC0/PDM claims and original source pages must be independently verified. No automatic training approval or bulk image downloading. Run cells in order. Enable **Internet** in Kaggle Notebook settings for Openverse and GitHub. CPU is sufficient for this preprocessing pipeline.

Outputs go to `/kaggle/working/mangaforge/` and can be downloaded as a Kaggle notebook output. To persist between sessions, save a notebook version and attach outputs as a Kaggle Dataset.


In [1]:
from pathlib import Path
import csv, json, hashlib, shutil, urllib.request, urllib.parse, urllib.error, time, re
from PIL import Image, ImageStat
ROOT=Path('/kaggle/working/mangaforge') if Path('/kaggle').exists() else Path.cwd()/'mangaforge'
ROOT.mkdir(parents=True,exist_ok=True)
SOURCE_DIR=Path('/kaggle/input/mangaforge-images')
REVIEWED_MANIFEST=Path('/kaggle/input/mangaforge-reviewed-manifest/manifest.csv')
REVIEWED_LABELS=Path('/kaggle/input/mangaforge-reviewed-labels/training_review.csv')
EXTS={'.png','.jpg','.jpeg','.webp'}
def readcsv(p):
    with p.open(encoding='utf-8-sig',newline='') as f:return list(csv.DictReader(f))
def writecsv(p,rows,fields):
    with p.open('w',encoding='utf-8',newline='') as f:
        w=csv.DictWriter(f,fieldnames=fields,extrasaction='ignore');w.writeheader();w.writerows(rows)
def yes(x):return str(x).strip().lower() in ('yes','true','1','ja')
print('Working directory:',ROOT)


Working directory: /kaggle/working/mangaforge


## 1. Find candidate images

Openverse queries return *metadata only*. Requests may be rate limited or unavailable. A failed request does not mean zero images exist. Search results are not automatically downloaded or approved.


In [2]:
# Robust Openverse metadata discovery: retry, checkpoint, and graceful offline fallback.
import socket, random
QUERIES=['manga drawing','comic illustration','ink drawing','samurai illustration','japanese landscape art','character sketch','comic action pose','fantasy illustration','woodblock print warrior','black white illustration','comic panel','magic illustration']
PAGES_PER_QUERY=2
CANDIDATES=ROOT/'openverse_candidates.csv'
CHECKPOINT=ROOT/'openverse_completed_pages.json'
FIELDS=['id','query','title','creator','license','license_url','source_url','image_url','width','height','attribution','commercial_allowed','ai_training_allowed','review_status']
rows=readcsv(CANDIDATES) if CANDIDATES.exists() else []
seen={r['id'] for r in rows}
try: completed=set(json.loads(CHECKPOINT.read_text(encoding='utf-8'))) if CHECKPOINT.exists() else set()
except (ValueError,OSError): completed=set()

def fetch_page(query,page):
    params=urllib.parse.urlencode({'q':query,'license':'cc0,pdm','page':page,'page_size':20,'mature':'false'})
    url='https://api.openverse.org/v1/images/?'+params
    for attempt in range(3):
        try:
            req=urllib.request.Request(url,headers={'User-Agent':'MangaForgeDatasetResearch/0.3 (metadata only)','Accept':'application/json'})
            with urllib.request.urlopen(req,timeout=25) as response:
                return json.load(response)
        except (urllib.error.HTTPError,urllib.error.URLError,TimeoutError,socket.timeout,ValueError,OSError) as exc:
            print(f'Attempt {attempt+1}/3 failed: {query} page {page}: {type(exc).__name__}: {str(exc)[:90]}',flush=True)
            if isinstance(exc,urllib.error.HTTPError) and exc.code in (400,401,403,404): break
            if attempt<2: time.sleep(min(4*(2**attempt),16)+random.random())
    return None

failures=0
for query in QUERIES:
    for page in range(1,PAGES_PER_QUERY+1):
        key=f'{query}::{page}'
        if key in completed: continue
        data=fetch_page(query,page)
        if data is None:
            failures+=1
            if failures>=3:
                print('Three consecutive failed pages. Stopping requests to avoid repeated timeouts. Existing CSV remains available.')
                break
            continue
        failures=0
        for item in data.get('results',[]):
            ident=str(item.get('id') or '')
            if not ident or ident in seen: continue
            seen.add(ident)
            rows.append({'id':ident,'query':query,'title':item.get('title') or '', 'creator':item.get('creator') or '', 'license':item.get('license') or '', 'license_url':item.get('license_url') or '', 'source_url':item.get('foreign_landing_url') or '', 'image_url':item.get('url') or '', 'width':item.get('width') or '', 'height':item.get('height') or '', 'attribution':item.get('attribution') or '', 'commercial_allowed':'no','ai_training_allowed':'no','review_status':'needs_manual_rights_review'})
        writecsv(CANDIDATES,rows,FIELDS)
        completed.add(key)
        CHECKPOINT.write_text(json.dumps(sorted(completed),ensure_ascii=False,indent=2),encoding='utf-8')
        print(f'{query} page {page}: {len(rows)} candidates total',flush=True)
        time.sleep(2)
    if failures>=3: break
if not CANDIDATES.exists(): writecsv(CANDIDATES,rows,FIELDS)
print('Candidates:',len(rows),'CSV:',CANDIDATES)
print('No candidates? Attach your own licensed image dataset; other pipeline steps can run without Openverse.')


Attempt 1/3 failed: manga drawing page 1: TimeoutError: The read operation timed out
Attempt 2/3 failed: manga drawing page 1: TimeoutError: The read operation timed out
Attempt 3/3 failed: manga drawing page 1: TimeoutError: The read operation timed out
Attempt 1/3 failed: manga drawing page 2: TimeoutError: The read operation timed out
Attempt 2/3 failed: manga drawing page 2: TimeoutError: The read operation timed out
Attempt 3/3 failed: manga drawing page 2: TimeoutError: The read operation timed out
Attempt 1/3 failed: comic illustration page 1: TimeoutError: The read operation timed out
Attempt 2/3 failed: comic illustration page 1: TimeoutError: The read operation timed out
Attempt 3/3 failed: comic illustration page 1: TimeoutError: The read operation timed out
Three consecutive failed pages. Stopping requests to avoid repeated timeouts. Existing CSV remains available.
Candidates: 0 CSV: /kaggle/working/mangaforge/openverse_candidates.csv
No candidates? Attach your own licensed

In [3]:
from pathlib import Path

ROOT = Path("/kaggle/working/mangaforge")
ROOT.mkdir(parents=True, exist_ok=True)

print("ROOT erfolgreich eingerichtet:", ROOT)

ROOT erfolgreich eingerichtet: /kaggle/working/mangaforge


## 2. Prepare local image inventory and verify rights

Upload images you own or have explicit permission to use into a Kaggle Dataset, attach it to the notebook, then point `SOURCE_DIR` to its folder under `/kaggle/input`. The inventory has `no` permissions by default. Copy the generated `manifest.csv` out, fill **source_url**, **license**, **commercial_allowed**, and **ai_training_allowed** based on evidence, and attach the reviewed manifest as a Kaggle Dataset for the next run. **Do not approve based on Openverse metadata alone.**


In [4]:
MANIFEST=ROOT/'manifest.csv'
fields=['path','sha256','license','source_url','permission_evidence','commercial_allowed','ai_training_allowed','review_status']
rows=[]
if SOURCE_DIR.is_dir():
    for p in sorted(SOURCE_DIR.rglob('*')):
        if not p.is_file() or p.suffix.lower() not in EXTS:continue
        try:
            with Image.open(p) as im: im.verify()
            digest=hashlib.sha256(p.read_bytes()).hexdigest()
            rows.append(dict(path=str(p),sha256=digest,license='',source_url='',permission_evidence='',commercial_allowed='no',ai_training_allowed='no',review_status='needs_manual_review'))
        except Exception as exc:print('Invalid image:',p.name,str(exc)[:90])
writecsv(MANIFEST,rows,fields)
print('Inventory:',len(rows),'Review CSV:',MANIFEST)


Inventory: 0 Review CSV: /kaggle/working/mangaforge/manifest.csv


In [5]:
APPROVED=ROOT/'approved_dataset';APPROVED.mkdir(exist_ok=True)
accepted=[]; rejected=[];seen=set()
if REVIEWED_MANIFEST.is_file():
    for row in readcsv(REVIEWED_MANIFEST):
        raw=row.get('path','');p=Path(raw)
        reason=''
        if not raw or not p.is_file() or p.suffix.lower() not in EXTS:reason='missing_or_invalid_path'
        elif not yes(row.get('commercial_allowed')) or not yes(row.get('ai_training_allowed')):reason='rights_not_approved'
        elif row.get('review_status','').strip().lower() not in ('approved','verified'):reason='review_not_approved'
        elif not row.get('permission_evidence','').strip():reason='missing_permission_evidence'
        else:
            digest=hashlib.sha256(p.read_bytes()).hexdigest()
            if row.get('sha256','').strip().lower()!=digest:reason='hash_mismatch'
            elif digest in seen:reason='duplicate'
            else:
                try:
                    with Image.open(p) as im:im.verify()
                    dst=APPROVED/(digest[:20]+p.suffix.lower());shutil.copy2(p,dst)
                    accepted.append({**row,'imported_path':str(dst)});seen.add(digest)
                except Exception:reason='invalid_image'
        if reason:rejected.append({**row,'reason':reason})
writecsv(APPROVED/'accepted.csv',accepted,fields+['imported_path'])
writecsv(APPROVED/'rejected.csv',rejected,fields+['reason'])
print('Accepted:',len(accepted),'Rejected:',len(rejected))


Accepted: 0 Rejected: 0


## 3. Quality audit and provisional classification

Technical audit uses dimensions, contrast and perceptual hashing. The provisional category below is inferred from the *filename*, not from an AI vision model. Manual visual review remains necessary.


In [6]:
quality=[]; review=[]
def category_for(name):
    name=name.lower()
    for word,cat in [('panel','comic_panel'),('face','portrait'),('character','character'),('landscape','background'),('action','action')]:
        if word in name:return cat
    return 'unclassified'
for row in accepted:
    p=Path(row['imported_path'])
    with Image.open(p) as im:
        w,h=im.size; gray=im.convert('L'); contrast=ImageStat.Stat(gray).stddev[0]
        tiny=gray.resize((8,8)); vals=list(tiny.getdata());avg=sum(vals)/len(vals)
        phash=''.join('1' if x>avg else '0' for x in vals)
    flag='review' if min(w,h)<512 or contrast<15 else 'pass'
    quality.append({'imported_path':str(p),'width':w,'height':h,'contrast':round(contrast,2),'a_hash':phash,'quality_status':flag})
    review.append({'imported_path':str(p),'category_suggestion':category_for(p.name),'category_verified':'','caption_verified':'','training_approved':'no'})
writecsv(APPROVED/'quality_report.csv',quality,['imported_path','width','height','contrast','a_hash','quality_status'])
writecsv(ROOT/'training_review.csv',review,['imported_path','category_suggestion','category_verified','caption_verified','training_approved'])
print('Quality audit:',len(quality),'Review labels:',ROOT/'training_review.csv')


Quality audit: 0 Review labels: /kaggle/working/mangaforge/training_review.csv


## 4. Training preparation (not training)

Only entries independently marked `training_approved=yes`, with a verified category and caption, enter `train.jsonl`. The notebook does not train a model or claim the output is legally cleared by itself. Use only a trusted, reviewed `training_review.csv`.


In [7]:
TRAIN=ROOT/'train.jsonl';count=0
approved_paths={str(Path(r['imported_path']).resolve()) for r in accepted}
quality_ok={r['imported_path'] for r in quality if r['quality_status']=='pass'}
with TRAIN.open('w',encoding='utf-8') as out:
    if REVIEWED_LABELS.is_file():
        for row in readcsv(REVIEWED_LABELS):
            raw=row.get('imported_path','');p=Path(raw)
            if not raw or str(p.resolve()) not in approved_paths or str(p) not in quality_ok or not p.is_file():continue
            if not yes(row.get('training_approved')):continue
            category=row.get('category_verified','').strip();caption=row.get('caption_verified','').strip()
            if not category or not caption:continue
            out.write(json.dumps({'image':str(p),'caption':caption,'category':category},ensure_ascii=False)+'\n');count+=1
print('Training records:',count,'JSONL:',TRAIN)


Training records: 0 JSONL: /kaggle/working/mangaforge/train.jsonl


### Notes
- Kaggle `/kaggle/input` is read-only. Output paths in `/kaggle/working` are temporary unless saved as notebook output/version.
- Image licenses, model training permissions, and publication rights are separate questions; consult the original rights holder and terms.
- Large-scale dataset work should be incremental, within Kaggle and source API limits.


## Export and GitHub integration
Download notebook and output files from Kaggle, then commit them to your repository. No GitHub credentials are required inside Kaggle. **Never commit private training images, tokens or permission documents.** GitHub integration is not performed automatically.


In [8]:
import zipfile
bundle=ROOT/'mangaforge_reports.zip'
with zipfile.ZipFile(bundle,'w',zipfile.ZIP_DEFLATED) as z:
    for f in [ROOT/'openverse_candidates.csv',ROOT/'manifest.csv',ROOT/'training_review.csv',ROOT/'train.jsonl',APPROVED/'accepted.csv',APPROVED/'rejected.csv',APPROVED/'quality_report.csv']:
        if f.is_file():z.write(f,f.relative_to(ROOT))
print('Download reports:',bundle)


Download reports: /kaggle/working/mangaforge/mangaforge_reports.zip
